In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-google-genai \
langchain-text-splitters \
langchain-chroma \
chromadb \
wikipedia-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 701.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61

In [2]:
import langchain
import chromadb
import wikipediaapi

print("All imports successful")

All imports successful


In [3]:
!pip list | grep chroma

chromadb                                 1.5.9
langchain-chroma                         1.1.0


In [4]:
import wikipediaapi
import os

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='rag-internship-project'
)

topics = [
    "Artificial intelligence",
    "Machine learning",
    "Deep learning",
    "Neural network",
    "Large language model",
    "Generative artificial intelligence",
    "Natural language processing",
    "Computer vision",
    "Reinforcement learning",
    "Retrieval-augmented generation"
]

os.makedirs("wiki_docs", exist_ok=True)

for topic in topics:
    page = wiki.page(topic)

    if page.exists():
        filename = topic.replace(" ", "_") + ".txt"

        with open(
            f"wiki_docs/{filename}",
            "w",
            encoding="utf-8"
        ) as f:
            f.write(page.text)

        print(f"Saved: {filename}")

    else:
        print(f"Page not found: {topic}")

Saved: Artificial_intelligence.txt
Saved: Machine_learning.txt
Saved: Deep_learning.txt
Saved: Neural_network.txt
Saved: Large_language_model.txt
Saved: Generative_artificial_intelligence.txt
Saved: Natural_language_processing.txt
Saved: Computer_vision.txt
Saved: Reinforcement_learning.txt
Saved: Retrieval-augmented_generation.txt


Verify Files

In [5]:
import os

files = os.listdir("wiki_docs")

print("Number of documents:", len(files))

for file in files:
    print(file)

Number of documents: 10
Natural_language_processing.txt
Neural_network.txt
Machine_learning.txt
Artificial_intelligence.txt
Large_language_model.txt
Retrieval-augmented_generation.txt
Generative_artificial_intelligence.txt
Reinforcement_learning.txt
Deep_learning.txt
Computer_vision.txt


**Ingestion** **Pipeline**

In [6]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader(
    "wiki_docs",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print("Documents loaded:", len(documents))

/tmp/ipykernel_7484/2037484324.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


Documents loaded: 10


check loaded data

In [7]:
print(documents[0].page_content[:1000])

Natural language processing (NLP) is the processing of natural language information by a computer. NLP is a subfield of computer science and is closely associated with artificial intelligence. NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly.
Major processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generation.

History
Natural language processing has its roots in the 1950s. Already in 1950, Alan Turing published an article titled "Computing Machinery and Intelligence," which proposed what is now called the Turing test as a criterion of intelligence, though at the time that was not articulated as a problem separate from artificial intelligence. The proposed test includes a task that involves the automated interpretation and generation of natural language.

Symbolic NLP (1950s – early 1990s)
The premise of symbolic NLP is oft

Download documents

In [8]:
!zip -r wiki_docs.zip wiki_docs

  adding: wiki_docs/ (stored 0%)
  adding: wiki_docs/Natural_language_processing.txt (deflated 64%)
  adding: wiki_docs/Neural_network.txt (deflated 59%)
  adding: wiki_docs/Machine_learning.txt (deflated 65%)
  adding: wiki_docs/Artificial_intelligence.txt (deflated 63%)
  adding: wiki_docs/Large_language_model.txt (deflated 63%)
  adding: wiki_docs/Retrieval-augmented_generation.txt (deflated 59%)
  adding: wiki_docs/Generative_artificial_intelligence.txt (deflated 61%)
  adding: wiki_docs/Reinforcement_learning.txt (deflated 72%)
  adding: wiki_docs/Deep_learning.txt (deflated 64%)
  adding: wiki_docs/Computer_vision.txt (deflated 65%)


In [9]:
from google.colab import files

files.download("wiki_docs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Chunk Documents

In [10]:
import langchain
print(langchain.__version__)

1.3.4


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500,
    chunk_overlap=250
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 245


In [12]:
print(chunks[0].page_content)

Natural language processing (NLP) is the processing of natural language information by a computer. NLP is a subfield of computer science and is closely associated with artificial intelligence. NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly.
Major processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generation.

History
Natural language processing has its roots in the 1950s. Already in 1950, Alan Turing published an article titled "Computing Machinery and Intelligence," which proposed what is now called the Turing test as a criterion of intelligence, though at the time that was not articulated as a problem separate from artificial intelligence. The proposed test includes a task that involves the automated interpretation and generation of natural language.

Symbolic NLP (1950s – early 1990s)
The premise of symbolic NLP is oft

In [13]:
print("Documents:", len(documents))
print("Chunks:", len(chunks))
print("Average chunks per document:", len(chunks)/len(documents))

Documents: 10
Chunks: 245
Average chunks per document: 24.5


**Embedding Model**

In [14]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get(
    "Gemini_key"
)

In [15]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(
    api_key=userdata.get("Gemini_key")
)

for model in genai.list_models():
    if "embed" in model.name.lower():
        print(model.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [16]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [17]:
from langchain_chroma import Chroma

db = Chroma(
    collection_name="test",
    embedding_function=embeddings,
    persist_directory="chroma_db"
)

print("Chroma initialized successfully")

Chroma initialized successfully


Store in ChromaDB

In [18]:
import chromadb
import langchain

print("chromadb:", chromadb.__version__)
print("langchain:", langchain.__version__)

chromadb: 1.5.9
langchain: 1.3.4


In [19]:
from langchain_chroma import Chroma
import time

persist_dir = "chroma_db"

vectorstore = None
batch_size = 20

for i in range(0, len(chunks), batch_size):

    batch = chunks[i:i+batch_size]

    print(f"Processing chunks {i+1} to {min(i+batch_size, len(chunks))}")

    if vectorstore is None:
        vectorstore = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            persist_directory=persist_dir,
            collection_name="wiki_rag"
        )
    else:
        vectorstore.add_documents(batch)

    print("✓ Batch stored successfully")

    time.sleep(35)

print("✓ All chunks stored in ChromaDB")

Processing chunks 1 to 20
✓ Batch stored successfully
Processing chunks 21 to 40
✓ Batch stored successfully
Processing chunks 41 to 60
✓ Batch stored successfully
Processing chunks 61 to 80
✓ Batch stored successfully
Processing chunks 81 to 100
✓ Batch stored successfully
Processing chunks 101 to 120
✓ Batch stored successfully
Processing chunks 121 to 140
✓ Batch stored successfully
Processing chunks 141 to 160
✓ Batch stored successfully
Processing chunks 161 to 180
✓ Batch stored successfully
Processing chunks 181 to 200
✓ Batch stored successfully
Processing chunks 201 to 220
✓ Batch stored successfully
Processing chunks 221 to 240
✓ Batch stored successfully
Processing chunks 241 to 245
✓ Batch stored successfully
✓ All chunks stored in ChromaDB


In [21]:
print("Documents stored:", vectorstore._collection.count())

Documents stored: 245


**Query Pipeline**

Retrieval

In [25]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

query = "What is machine learning?"

results = retriever.invoke(query)

print(f"Retrieved {len(results)} chunks")

Retrieved 5 chunks


In [23]:
try:
    print("Vectorstore docs:", vectorstore._collection.count())
except:
    print("vectorstore missing")

try:
    print("Retriever exists")
    print(type(retriever))
except:
    print("retriever missing")

Vectorstore docs: 245
Retriever exists
<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


In [26]:
for i, doc in enumerate(results, start=1):
    print("=" * 50)
    print(f"Chunk {i}")
    print(doc.page_content[:500])
    print()

Chunk 1
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statistics and mathematical optimisation methods compose the foundations o

Chunk 2
Models
A machine learning model is a type of mathematical model that, once "trained" on a given dataset, can be used to make predictions or classifications on new data. During training, a learning algorithm iteratively adjusts the model's internal parameters to minimise errors in its predictions. By extension, the term "model" can refer to several levels of specificity, from a general class of models and their associated learning algorithms to a fully trained model with all its

Answer Generation

In [31]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.3 MB/s eta 0:00:00


In [32]:
from groq import Groq
from google.colab import userdata

groq_api_key = userdata.get("Groq_Key")

client = Groq(api_key=groq_api_key)

print("Groq initialized successfully")

Groq initialized successfully


In [33]:
context = "\n\n".join(
    [doc.page_content for doc in results]
)

print(context[:1000])

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.
From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimisation under this framework.

Models
A machine learning model is a type of mathematical model that, once "t

In [34]:
prompt = f"""
You are a helpful assistant.

Answer ONLY using the provided context.

If the answer is not present in the context,
say "I could not find that information in the documents."

Context:
{context}

Question:
{query}
"""

In [35]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response.choices[0].message.content

print(answer)

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.


create function

In [36]:
def ask_question(query):

    docs = retriever.invoke(query)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
    You are a helpful assistant.

    Answer ONLY using the provided context.

    If the answer is not present in the context,
    say "I could not find that information in the documents."

    Context:
    {context}

    Question:
    {query}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

test function

In [37]:
answer = ask_question(
    "What is machine learning?"
)

print(answer)

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.


In [38]:
answer = ask_question(
    "What are the types of machine learning?"
)

print(answer)

The types of machine learning approaches are traditionally divided into three broad categories: 

1. Supervised learning: The computer is presented with example inputs and their desired outputs, given by a "teacher", and the goal is to learn a general rule that maps inputs to outputs.
2. Unsupervised learning: No labels are given to the learning algorithm, leaving it on its own to find structure in its input.
3. Reinforcement learning: A computer program interacts with a dynamic environment in which it must perform a certain goal and is provided rewards as feedback, which it tries to maximize, thus resulting in the program learning from experience.

Additionally, there are other types of machine learning, including:
- Semi-supervised learning: falls between unsupervised learning and supervised learning, where some of the training examples are missing training labels.
- Self-learning: a machine learning paradigm that gives a solution to the problem of learning without any external rewar

In [39]:
answer = ask_question(
    "Who is Alan Turing?"
)

print(answer)

Alan Turing is a mathematician and philosopher who investigated whether machines can show intelligent behavior and think. In 1950, he proposed the Turing test, which measures the ability of a machine to simulate human conversation. He also introduced the theory of computation, which suggested that a machine could simulate any conceivable form of mathematical reasoning by shuffling symbols as simple as "0" and "1". Additionally, he wrote an influential 1950 paper 'Computing Machinery and Intelligence', which introduced the concept of machine intelligence.


In [40]:
answer = ask_question(
    "What is computer vision?"
)

print(answer)

Computer vision is an interdisciplinary field that deals with how computers can be made to gain high-level understanding from digital images or videos. It seeks to automate tasks that the human visual system can do, and involves the development of a theoretical and algorithmic basis to achieve automatic visual understanding.
